# Where-fi - Object localization using RSSI of Wi-fi signals indoors

### Project for the INTAR lecture at UPV

## Imports and configurations

### Imports of useful libraries

In [ ]:
# Libraries for logging and general utilities
import random
import logging
import wandb
import time
import math
import yaml
from box import Box

# Libraries for data manipulation and analysis
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

# Helpful libraries
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, Optional, Tuple, Union

# Deep learning libraries
import torch
from torch import nn
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
import torchvision.transforms as T
from torchmetrics.classification import (AUROC, Accuracy, ConfusionMatrix,
                                         F1Score, MulticlassAUROC, Precision,
                                         Recall)

### Configuration to control model architecture and training

In [ ]:
%%writefile config.yaml
seed: 42
training:
    batch_size: 64
    num_epochs: 50
    learning_rate: 0.001
    learning_rate_scheduler: "Cosine"
    learning_rate_min: 1e-6
    beta1: 0.9
    beta2: 0.999
    weight_decay: 1e-2
logging:
    use_wandb: True
    main_dir: "Path to your main directory here"
    log_intervall: 5
    eval_intervall: 5
    team_name: "UPV_projects"
    project_name: "Where-fi"
    api_key: null
    # Add-on for the run name
    run_name_additional: null
model:
    name: "YourModelNameHere"
dataset:
    root_path: ""
    training_data_name: "TrainingData.csv"
    validation_data_name: "ValidationData.csv"
    no_signal_value: -110
    signal_max: 0
    longitude_min: -7695.9387549299299000
    longitude_max: -7299.786516730871000
    latitude_min: 4864745.7450159714
    latitude_max: 4865017.3646842018
data_augmentation:
    noise:
        enabled: True,
        noise_level: 0.091
    dropout:
        enabled: True,
        probability: 0.01
    shift:
        enabled: True,
        max_shift: 0.0455,
        probability: 0.3

In [ ]:
# To access by CONFIG.training.batch_size instead of CONFIG['training']['batch_size']
with open("config.yaml", "r") as f:
    cfg = Box(yaml.safe_load(f))

## Helper classes
Classes for utility functionalities

### Class for training parameters
Small helper class to organise the parameters for the training

In [ ]:
class TrainingParameters:
    '''
    Class to encapsulate training parameters.
    Contains learning rate, beta values, weight decay, and number of epochs.
    '''

    def __init__(self,
            lr: float,
            lr_min: float,
            lr_scheduler: str,
            beta1: float,
            beta2: float,
            weight_decay: float,
            epochs: int
        ):
        '''
        Args:
            lr (float): Learning rate
            beta1 (float): Beta1 value for optimizer
            beta2 (float): Beta2 value for optimizer
            weight_decay (float): Weight decay for optimizer
            epochs (int): Number of training epochs

        Initializes the training parameters with the given values.
        '''
        self.lr = lr
        self.lr_min = lr_min
        self.lr_scheduler = lr_scheduler
        self.beta1 = beta1
        self.beta2 = beta2
        self.weight_decay = weight_decay
        self.epochs = epochs

    def get_learning_rate(self) -> float:
        '''
        Get the learning rate.
        '''
        return self.lr

    def get_learning_rate_min(self) -> float:
        '''
        Get the minimum learning rate.
        '''
        return self.lr_min

    def get_betas(self) -> tuple[float, float]:
        '''
        Get the beta values.
        '''
        return (self.beta1, self.beta2)

    def get_weight_decay(self) -> float:
        '''
        Get the weight decay.
        '''
        return self.weight_decay

    def get_epochs(self) -> int:
        '''
        Get the number of epochs.
        '''
        return self.epochs

    def get_lr_scheduler(self) -> str:
        '''
        Get lr_schedular
        '''
        return self.lr_scheduler

## Dataset

### Dataaugmentation

In [ ]:
class GaussianNoise:
    '''
    Class to add Gaussian noise to the input signal.
    '''
    def __init__(self, std_range=(0.0, 0.1), p=0.0):
        """
        std: relative noise strength
        p: probability of applying the noise
        """
        self.std_range = std_range
        self.p = p

    def __call__(self, x):
        """
        x: (B, C)
        """
        if random.random() < self.p:
            std = random.uniform(self.std_range[0], self.std_range[1])
            noise = torch.randn_like(x) * std
            # Maximal noise of given level
            noise = torch.clamp(noise, -0.1, 0.1)
            result = x + noise
            # Make sure the result is still between 0 and 1
            result = torch.clamp(result, 0.0, 1.0)
            return result
        return x

In [ ]:
class RandomValueDropout:
    '''
    Class to randomly drop (set to zero) some values in the input signal.
    '''
    def __init__(self, p=0.0):
        '''
        p: probability of dropping a value.
        '''
        self.p = p

    def __call__(self, x):
        '''
        Args:
            x (torch.Tensor): Input tensor of shape (B, C).
        Returns:
            torch.Tensor: Tensor with some values dropped.
        '''
        if random.random() < self.p:
            mask = torch.rand_like(x) > 0.9
            x = x.masked_fill(mask, 0.0)
        return x

In [ ]:
class Shifting:
    '''
    Class to add a shift to the input signal.
    '''
    def __init__(self, shift=0.05, p=0.0):
        """
        shift: relative shift strength
        p: probability of applying the shift
        """
        self.shift = shift
        self.p = p

    def __call__(self, x):
        """
        x: (B, C)
        """
        if random.random() < self.p:
            shift = random.uniform(-self.shift, self.shift)
            # Apply the shift
            result = x + shift
            # Make sure the result is still between 0 and 1
            result = torch.clamp(result, 0.0, 1.0)
            return result
        return x

In [ ]:
class DataAugmentation:
    """
    Class for randomly applying data augmentations of the following list:
        - Noise
        - Null masking of RSS values
        - Bias shifting of RSS values
    """
    def __init__(self, cfg: any):
        '''
        Initializes the DataAugmentation with a composition of random
        transformations.
        '''
        self.transform = T.Compose([
            GaussianNoise(std_range=(0.0, cfg.data_augmentation.noise.noise_level), p=cfg.data_augmentation.noise.probability),
            Shifting(shift=cfg.data_augmentation.shift.max_shift, p=cfg.data_augmentation.shift.probability),
            RandomValueDropout(p=cfg.data_augmentation.dropout.probability)
        ])

    def __call__(self, x: torch.Tensor) -> torch.Tensor:
        '''
        Args:
            x (torch.Tensor): Input tensor to be augmented.
        Returns:
            torch.Tensor: Augmented tensor.

        Method to apply the random data augmentations to the input signal.
        '''
        return self.transform(x)

### Dataset

In [ ]:
class ProjectDataset(Dataset):
    """
    Dataset class for loading ...
    """
    def __init__(self, cfg: Any,
                 split: str,
                 sample_transform: transforms = None,
                 label_transform: transforms = None
        ) -> None:
        """
        Initialize the dataset.

        Args:
            cfg: Configuration object with dataset paths and parameters.
            split: One of 'train', 'val', or 'test'.
            scaler: Optional dictionary {'mean': ..., 'std': ...} for normalization.
        """
        self.split = split
        self.random_seed = cfg.seed
        self.cfg = cfg.dataset
        self.data_root = Path(self.cfg.root_path)


        training_data_path = self.data_root / self.cfg.training_data_name
        test_data_path = self.data_root / self.cfg.validation_data_name

        # Check if files exist
        if not training_data_path.exists():
            raise FileNotFoundError(f"Training data file not found at {training_data_path}")
        if not test_data_path.exists():
            raise FileNotFoundError(f"Test data file not found at {test_data_path}")


        self.samples, self.labels = self._load_data()

        if len(self.samples) == 0:
            raise RuntimeError(
                f"Dataset split '{split}' is empty. "
            )


    def _load_data(self) -> Tuple[np.ndarray, np.ndarray]:
        """
        Load raw sensor files, merge them, and segment into datapoints.
        
        Returns:
            Tuple of (samples, labels) where samples is (N, Window, 6).
        """
        # Remove unnecessary columns
        drop_columns = ["RELATIVEPOSITION", "USERID", "PHONEID", "TIMESTAMP"]
        
        if self.split == 'test':
            # Read test data
            test_data = pd.read_csv(self.test_data_path)
            data = test_data.drop(columns=drop_columns)

        elif self.split in ['training', 'validation']:
            # Read training & validation data
            training_data = pd.read_csv(self.training_data_path)
            training_data = training_data.drop(columns=drop_columns)

            # Split into training & validation data
            training_data, validation_data = train_test_split(
                training_data, 
                test_size=0.10,           # 10% for validation
                random_state=self.random_seed,          # For deterministic
                stratify=training_data['class_id']  # Guarantees same representation of classes
            )

            if self.split == 'training':
                data = training_data
            else:
                data = validation_data
        else:
            raise ValueError(f"Invalid split '{self.split}'. Must be one of 'training', 'validation', or 'test'.")


        # Replace no signal values with a specific value (e.g., -110 dBm)
        data = data.replace(100, self.cfg.no_signal_value)

        # Normalize
        columns_to_normalize = training_data.columns.difference(['FLOORID', 'BUILDINGID', 'LATITUDE', 'LONGITUDE'])
        max_value = 0
        min_value = self.cfg.no_signal_value
        data[columns_to_normalize] = (data[columns_to_normalize] - min_value) / (max_value - min_value)

        # Organize labels
        # Append building ID and floor ID to create a unique label for each location
        data['class_id'] = data['BUILDINGID'].astype(str) + '_' + data['FLOORID'].astype(str)

        data = data.drop(columns=['BUILDINGID', 'FLOORID'])

        # Check whether longitude or lattitude values are outside of the expected range and raise an error if so
        if (data['LONGITUDE'] < self.cfg.longitude_min).any() or (data['LONGITUDE'] > self.cfg.longitude_max).any():
            raise ValueError("Longitude values are outside of the expected range.")
        if (data['LATITUDE'] < self.cfg.latitude_min).any() or (data['LATITUDE'] > self.cfg.latitude_max).any():
            raise ValueError("Latitude values are outside of the expected range.")

        # Normalize longitude and latitude to be between 0 and 1
        data['LATITUDE'] = (data['LATITUDE'] - self.cfg.latitude_min) / (self.cfg.latitude_max - self.cfg.latitude_min)
        data['LONGITUDE'] = (data['LONGITUDE'] - self.cfg.longitude_min) / (self.cfg.longitude_max - self.cfg.longitude_min)

        label_columns = ["class_id", "LATITUDE", "LONGITUDE"]
        labels = data[label_columns]
        samples = data.drop(columns=label_columns)

        return np.array(samples, dtype=np.float32), np.array(labels)


    def __len__(self) -> int:
        """
        Return the number of valid windows in the dataset.
        """
        return len(self.samples)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, Union[int, torch.Tensor]]:
        """
        Return a window and its label.
        
        Output shapes:
        - seq2label: (Window_Size, 6), Scalar Integer
        - seq2seq:   (Window_Size, 6), (Window_Size,) (LongTensor)
        """
        # TODO: Adapt to projects needs
        x = torch.from_numpy(self.samples[idx])
        y = self.labels[idx]

        if self.sample_transform:
            x = self.sample_transform(x)
        if self.label_transform:
            y = self.label_transform(y)

        return x, y

## Class for general utilities

In [ ]:
class Utilities:
    '''
    Class to encapsulate utilities for training, such as logging.
    '''

    def __init__(self):
        '''
        Class to organize different utility functionalities.
        '''
        pass

    def seed_worker(self, worker_id):
        '''
        Seed worker for DataLoader to ensure reproducibility.
        '''
        _ = worker_id  # Unused, but required by the DataLoader
        worker_seed = torch.initial_seed() % 2**32
        np.random.seed(worker_seed)
        random.seed(worker_seed)

    def get_dataset(self, cfg, split='train', scaler=None):
        '''
        Args:
            cfg: Config object
            split: 'train', 'val', or 'test'
            scaler: The scaler dict {'mean':..., 'std':...} from the train set.
                    Only required if split is 'val' or 'test'.
        
        Return:
            dataset
            Dataloader
            current_scaler: scalar stats of the training Data
        '''
        if split == 'train':
            transform_ops = [
                DataAugmentation(cfg.dataset)
            ]
        else:  # no augmentation for test
            transform_ops = [
            ]

        datapoint_transform = T.Compose(transform_ops)
        label_transform = T.Compose([])


        dataset = ProjectDataset(cfg, split, scaler=scaler, sample_transform=datapoint_transform,
                        label_transform=label_transform)


        # Generator for reproducibility
        g = torch.Generator()
        g.manual_seed(cfg.seed)

        dataloader = DataLoader(
            dataset, 
            batch_size=cfg.dataset.batch_size, 
            shuffle=(split == 'train'), # Just shuffle during training
            worker_init_fn=self.seed_worker,
            generator=g
        )

        return dataset, dataloader


    def set_seed(self, seed: int):
        '''
        Set random seed for reproducibility across all libraries.

        Args:
            seed (int): Random seed value
        '''
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)  # For multi-GPU
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

    def get_parameters_table(self, model: torch.nn.Module, position: int = 0) -> str:
        '''
        Args:
            model (torch.nn.Module): The PyTorch model to summarize
            position (int): The position in the model hierarchy (recursive depth)

        Returns a list of tuples containing (module name, position, number of trainable parameters).
        '''
        # name, position, number trainable parameters
        lines = []
        num_trainable_params = 0
        for _, param in model.named_parameters(recurse=False):
            if param.requires_grad:
                num_trainable_params += math.prod(param.size())

        child_lines = []
        for module in model.children():
            child_lines += self.get_parameters_table(module, position + 1)

        sum_params = 0
        for line in child_lines:
            if line[1] == position + 1:
                sum_params += line[2]

        if num_trainable_params == 0:
            lines.append([type(model).__name__, position, sum_params])
        else:
            lines.append([type(model).__name__, position, num_trainable_params])
        lines += child_lines

        return lines

    def get_model_summary(self, model: torch.nn.Module) -> str:
        '''
        Args:
            model (torch.nn.Module): The PyTorch model to summarize

        Returns a string representation of the model's summary.
        '''
        lines = self.get_parameters_table(model)
        summary = ""
        for line in lines:
            summary += "  " * line[1] + f"└─ {line[0]}: {line[2]} trainable parameters\n"
        summary += "=================================================================\n"
        summary += f"Total Trainable Parameters: {lines[0][2]}\n"
        return summary

## Evaluator class
Class for handling the evaluation of the model with the test set, calculating different relevant metrics

## Trainer class
Class for handling the training of the model

In [ ]:
class Trainer:
    '''
    Class to handle the training loop for a model.
    '''

    def __init__(
        self,
        cfg: any, # Contains the configuration parameters for training, such as learning rate, epochs, etc.
        train_loader: DataLoader,
        model: Module,
        evaluator: Evaluator,
        device: torch.device,
        wb_run: Any
    ):
        '''
        Args:
            cfg (object): Configuration object with training parameters.
            train_loader (DataLoader): DataLoader for the training dataset.
            model (nn.Module): The model to be trained.
            evaluator (Evaluator): Evaluator for validation during training.
            device (torch.device): Device to run the training on.

        Initializes the Trainer with the given arguments.
        '''
        self.train_loader = train_loader

        # Ogranize training parameters, so not all are attributes of this class
        self.training_parameters = TrainingParameters(
            lr=cfg.model.lr,
            lr_min=cfg.model.lr_min,
            lr_scheduler=cfg.model.lr_scheduler,
            beta1=cfg.beta1,
            beta2=cfg.beta2,
            weight_decay=cfg.weight_decay,
            epochs=cfg.epochs,
        )
        
        # Create optimizer
        optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=self.training_parameters.get_learning_rate(),
            betas=self.training_parameters.get_betas(),
            weight_decay=self.training_parameters.get_weight_decay()
        )

        # Initialize training components
        model.to(device)
        self.model = model
        self.optimizer = optimizer
        self.evaluator = evaluator
        self.device = device

        # Set intervals
        self.log_interval = cfg.log_interval
        self.eval_interval = cfg.eval_interval

        # Set loss function
        self.criterion = torch.nn.CrossEntropyLoss(ignore_index=0)

        # Change learning rate scheduler based on config
        self.scheduler = None
        if self.training_parameters.lr_scheduler == "cosine":
            self.scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
                self.optimizer,
                T_max=self.training_parameters.get_epochs(),
                eta_min=self.training_parameters.get_learning_rate_min()
            )

        self.wb_run = wb_run


    def get_trained_model(self) -> Module:
        '''
        Returns:
            The trained model

        Function to access the trained model by the Trainer class.
        '''
        return self.model

    def train(self):
        '''
        Main training loop.
        '''
        for epoch in range(self.training_parameters.get_epochs() - self.start_epochs):
            start_time = time.time()
            # Set model to training mode
            self.model.train()
            running_loss = self.start_loss
            correct = 0
            total = 0

            for step, (x, y_true) in enumerate(self.train_loader):
                x, y_true = x.to(self.device), y_true.to(self.device)

                self.optimizer.zero_grad()

                y_pred = self.model(x)

                if self.mode == "seq2seq":
                    B, T, C = y_pred.shape # batch, time, classes
                    y_pred = y_pred.reshape(B * T, C)
                    y_true = y_true.reshape(B * T)

                loss = self.criterion(y_pred, y_true)

                loss.backward()

                # Gradient clipping 
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)

                self.optimizer.step()

                running_loss += loss.item() * x.size(0)
                predicted_classes = torch.argmax(y_pred, dim=1) 
                valid_mask = y_true != 0                

                total += valid_mask.sum().item()
                correct += ((predicted_classes == y_true) & valid_mask).sum().item()

                if self.wb_run is not None and ((step + 1) % self.log_interval == 0):
                    accuracy = correct / total if total > 0 else 0
                    # Log the base learning rate (first param group, likely head)
                    current_lr = self.optimizer.param_groups[0]['lr']

                    self.wb_run.log({
                        "accuracy_training": accuracy,
                        "loss_training": loss.item(),
                        "learning_rate": current_lr
                    })

            end_time = time.time()
            if self.wb_run is not None:
                self.wb_run.log({"epoch_duration": end_time - start_time})

            if self.scheduler is not None:
                self.scheduler.step()

            if (epoch + 1) % self.eval_interval == 0:
                # Set model to evaluation mode
                self.model.eval()
                with torch.no_grad():
                    self.evaluator.eval()

        logging.info("Training finished")

In [ ]:
class Evaluator:
    """
    Evaluator for binary and multiclass classification tasks.

    Handles model evaluation with metrics computation for both:
        - Binary classification: num_classes=1 (BCELoss) or num_classes=2 (CrossEntropyLoss)
        - Multiclass classification: num_classes>2 (CrossEntropyLoss)

    Automatically selects metrics (Accuracy, Precision, Recall, F1, AUROC, Confusion Matrix)
    based on the classification task type.
    """

    def __init__(
        self,
        cfg: Any,
        eval_loader: DataLoader,
        model: nn.Module,
        device: torch.device,
        wb_run: Any,
        criterion: nn.Module = None,

    ):
        """
        Args:
            cfg: Hydra config. Must contain 'num_classes'.
            eval_loader (DataLoader): DataLoader for the evaluation dataset.
            model (nn.Module): The model to evaluate.
            device (torch.device): 'cuda' or 'cpu'.
            criterion (nn.Module, optional): Loss function (nn.BCELoss or nn.CrossEntropyLoss).
                                            Uses 'nn.CrossEntropyLoss' as default.
        """
        self.cfg = cfg
        self.eval_loader = eval_loader
        self.model = model
        self.device = device
        self.wb_run = wb_run
        self.mode = cfg.mode

        self.criterion = criterion if criterion is not None else nn.CrossEntropyLoss()

        # Raise error if cfg doesnt have 'num_classes' attribute
        if not hasattr(cfg, 'num_classes'):
            raise ValueError(
                "Config object 'cfg' must contain 'num_classes' attribute.")

        self.num_classes = cfg.num_classes
        self.use_bce = isinstance(self.criterion, nn.BCELoss)

        # Define a dict containing "task" and "num_classes" for intializing the metrics
        if self.num_classes <= 2:
            self.task = "binary"
            # No num_classes needed for binary
            self.task_kwargs = {"task": self.task}
        else:  # num_classes > 2
            self.task = "multiclass"
            self.task_kwargs = {"task": self.task,
                                "num_classes": self.num_classes}

        # Creates and fills a dict to store all metrics
        self.metrics = nn.ModuleDict()
        self.metrics.update(self._get_metrics())
        for metric in self.metrics.values():
            metric.to(device)

    def _get_metrics(self) -> dict:
        """
        Build the dictionary of evaluation metrics.

        Binary metrics:
            - Accuracy, Precision, Recall, F1, AUROC, Confusion Matrix

        Multiclass metrics:
            - Accuracy (micro-averaged)
            - Precision, Recall, F1, AUROC (macro-averaged)
            - Confusion Matrix

        Returns:
            dict: Dictionary mapping metric names to torchmetrics instances.
        """

        # Metrics that are common in both binary and multi
        metrics = {
            'conf_matrix': ConfusionMatrix(**self.task_kwargs),
        }

        # Define task specific metrics
        if self.task == "binary":
            metrics.update({
                'accuracy_evaluation': Accuracy(**self.task_kwargs),
                'auroc': AUROC(**self.task_kwargs),
                'precision': Precision(**self.task_kwargs),
                'recall': Recall(**self.task_kwargs),
                'f1': F1Score(**self.task_kwargs),
            })
        else:  # multiclass
            # macro gives each class the same weight, micro each sample
            macro_kwargs = {**self.task_kwargs, "average": "macro"}
            micro_kwargs = {**self.task_kwargs, "average": "micro"}
            metrics.update({
                'accuracy_evaluation': Accuracy(**micro_kwargs),
                'auroc_macro': AUROC(**macro_kwargs),
                'precision_macro': Precision(**macro_kwargs),
                'recall_macro': Recall(**macro_kwargs),
                'f1_macro': F1Score(**macro_kwargs),
            })
        return metrics

    def _reset_metrics(self) -> None:
        """
        Resets all metric states. Called at the beginning of eval.
        """
        for metric in self.metrics.values():
            metric.reset()

    @torch.no_grad()
    def eval(self, return_metrics: bool = False) -> None:
        """
        Args:
            return_metrics: Optionally return metric. Default: False

        Run a full evaluation loop over the evaluation dataset and logs the metrics to
        Weights & Biases.
        """
        self.model.eval()
        self._reset_metrics()

        running_loss = 0.0

        for _, (x, y_true) in enumerate(self.eval_loader):
            x, y_true = x.to(self.device), y_true.to(self.device)

            # Make a prediction
            y_pred = self.model(x)  # Model ouput

            if self.mode == "seq2seq":
                B, T, C = y_pred.shape  # batch, time, classes
                y_pred = y_pred.reshape(B * T, C)
                y_true = y_true.reshape(B * T)

            if self.use_bce:
                # Uses model output AFTER SIGMOID
                loss = self.criterion(y_pred, y_true)
                y_pred_probs = y_pred.squeeze()  # Shape [N]
                y_pred_labels = (y_pred_probs > 0.5).int()  # Shape [N]
                y_true = y_true.squeeze()  # Shape [N]
            elif isinstance(self.criterion, nn.CrossEntropyLoss):
                # Uses model output BEFORE SOFTMAX
                loss = self.criterion(y_pred, y_true)
                y_pred_probs = torch.softmax(y_pred, dim=1)  # Shape [N, C]
                y_pred_labels = torch.argmax(y_pred_probs, dim=1)  # Shape [N]
            else:
                # Fallback or error for unsupported criteria
                raise NotImplementedError(
                    f"Unsupported criterion {type(self.criterion)}; "
                    "use nn.BCELoss or nn.BCEWithLogitsLoss."
                )

            running_loss += loss.item()

            # Iterate through the dict to update the metrics
            for metric_name, metric in self.metrics.items():
                if isinstance(metric, AUROC) or isinstance(metric, MulticlassAUROC):
                    # auroc uses probability instead of labels
                    metric.update(y_pred_probs, y_true)
                else:
                    metric.update(y_pred_labels, y_true)

        # Compute final metrics for hole dataset after iterating through all batches
        final_metrics = {}

        # Calculate average loss per batch
        final_metrics = {
            'loss_evaluation': running_loss / len(self.eval_loader)}

        # Compute all other metrics
        for metric_name, metric in self.metrics.items():
            try:
                final_metrics[metric_name] = metric.compute()
            except Exception as e:
                logging.warning("[Warning] Could not compute metric '%s'. Set to None. Error: %s",
                                metric_name, e)
                final_metrics[metric_name] = None

        if return_metrics:
            return final_metrics

## Model
Class of the selected model including the definition of the architecture as well as the forward porapagation

In [ ]:
class ModelName:
    def __init__(self, cfg: Any):
        '''
        Initializes the model based on the configuration.
        '''
        self.name = "ModelName"

        # TODO: Implement model architecture based on cfg.model
        
        pass

    def get_name(self) -> str:
        '''
        Returns the name of the model.
        '''
        return self.name

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        '''
        Forward pass of the hybrid model.

        Args:
            x: Input tensor

        Returns:
            Output tensor after passing through the model.
        '''

        # TODO: Implement the forward pass of the model

        return x

## Main function

In [ ]:
def main(config=None):
    '''
    Main function to run the training and evaluation pipeline.
    '''
    if config is None:
        raise ValueError("Config dictionary must be provided.")
    
    cfg = CONFIG
    
    # Create utility instance
    util = Utilities()
    
    # Set seed for reproducibility
    util.set_seed(cfg.seed)

    # Select device (GPU if available, else CPU)
    device = torch.device('cpu')
    if torch.cuda.is_available():
        device = torch.device('cuda')
    elif torch.backends.mps.is_available():
        device = torch.device('mps')
    logging.info('Using device: %s', device)

    # Get Datasets
    _, train_dataloader = util.get_dataset(cfg, split='train')
    _, eval_dataloader, _ = util.get_dataset(cfg, split='val')
    _, test_dataloader, _ = util.get_dataset(cfg, split='test')

    # Get Model
    model = ModelName(cfg.model)
    model_summary = util.get_model_summary(model)
    logging.info(model_summary)

    # Start a new wandb run to track this script
    if cfg.logging.use_wandb:
        timestamp = datetime.now().strftime('%Y%m%d_%H%M')
        wandb.login(key=cfg.logging.api_key)
        additional = cfg.logging.run_name_additional
        if additional != 'None' and additional is not None:
            run_name = f"{model.get_name()}_{additional}_{cfg.dataset.name}_{timestamp}"
        else:
            run_name = f"{model.get_name()}_{cfg.dataset.name}_{timestamp}"
        wb_run = wandb.init(
            # Set the wandb entity where your project will be logged
            entity=cfg.logging.team_name,
            # Set the wandb project where this run will be logged
            project=cfg.logging.project_name,
            # Name of the run
            name=run_name,
            # Track hyperparameters and run metadata
            config={
                "learning_rate": cfg.model.lr,
                "learning_rate scheduler": cfg.model.lr_scheduler,
                "architecture": model.get_name(),
                "dataset": cfg.dataset.name,
                "epochs": cfg.epochs,
                # Log all model config settings
                "model settings": {**dict(cfg.model)},
                "model architecture": model_summary
            }
        )
    else:
        wb_run = None


    # Get evaluator
    evaluator = Evaluator(cfg, eval_dataloader, model, device, wb_run)

    # Get trainer
    trainer = Trainer(cfg,
                      train_dataloader,
                      model,
                      evaluator,
                      device,
                      wb_run)

    # Train the model and perform final evaluation on test set
    trainer.train()

    test_evaluator = Evaluator(cfg, test_dataloader, trainer.get_trained_model(), device, criterion=None)
    logging.info("Running Evaluation")
    test_evaluator.eval()


if __name__ == '__main__':
    main()
